# Thabtah Combined — ages **4–11** (AQ-10-Child screening aid)

This notebook trains the **webpage questionnaire model** for children aged **4–11**.

**What this is:** a screening aid. The site collects **AQ-10-Child** answers (A1–A10) and returns screen-positive / screen-negative plus a probability. It is **not** a diagnosis.

**What this is not:** a search for higher accuracy. Public tables already label YES/NO from the same 10 questions, so a strong items-only score means we recovered the official cutoff. Sensitive clinic labels are not openly available.

| File | Rows | Ages | Use |
|---|---:|---|---|
| `asd_screening_thabtah_combined_age4to11_years.csv` | ~960 after dedup | **4–11** | **this notebook / `.pkl` for the app** |
| `…_age1to11_years.csv` | ~1,735 | 1–11 | **do not train** — toddlers use Q-CHAT-10, not AQ-10 |

Do not merge with toddler months files. Different instrument.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay,
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
ITEM_COLS = [f"A{i}_Score" for i in range(1, 11)]

## 0. Load the 4–11 CSV

Looks in Colab `/content/`, the current folder, then the repo `data/raw/questionnaire/` path.

In [ ]:
candidates = [
    Path("/content/asd_screening_thabtah_combined_age4to11_years.csv"),
    Path("asd_screening_thabtah_combined_age4to11_years.csv"),
    Path("../data/raw/questionnaire/asd_screening_thabtah_combined_age4to11_years.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Upload asd_screening_thabtah_combined_age4to11_years.csv or run "
        "scripts/build_thabtah_combined_age4to11_csv.py locally"
    )

df = pd.read_csv(data_path)
df["class"] = df["class"].astype(str).str.strip().str.upper()
print("Loaded", data_path.resolve())
print("shape", df.shape)
print("columns", list(df.columns))
df.head()

## 1. Data pre-analysis

Check size, missing values, class balance, age band, and background fields **before** fitting anything. The webpage only needs A1–A10; demographics are an audit (they should not drive the app).

In [ ]:
print("dtypes\n", df.dtypes)
print("\nmissing per column\n", df.isna().sum())
print("\nclass counts\n", df["class"].value_counts())
print("class %\n", (df["class"].value_counts(normalize=True) * 100).round(1))
print("\nage years: min", df["age"].min(), "max", df["age"].max(), "median", df["age"].median())
print("unique ages", sorted(df["age"].dropna().unique()))

for col in ["gender", "jaundice", "autism"]:
    if col in df.columns:
        print(f"\n{col}\n", df[col].astype(str).str.strip().str.lower().value_counts(dropna=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.countplot(data=df, x="class", ax=axes[0], order=["NO", "YES"])
axes[0].set_title("Class balance (screen-negative / screen-positive)")
sns.histplot(data=df, x="age", hue="class", multiple="dodge", discrete=True, ax=axes[1])
axes[1].set_title("Age (years) by class")
sns.countplot(data=df, x="gender", hue="class", ax=axes[2])
axes[2].set_title("Gender by class")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
if "jaundice" in df.columns:
    sns.countplot(data=df, x="jaundice", hue="class", ax=axes[0])
    axes[0].set_title("Jaundice by class")
if "autism" in df.columns:
    sns.countplot(data=df, x="autism", hue="class", ax=axes[1])
    axes[1].set_title("Family ASD (autism column) by class")
plt.tight_layout()
plt.show()

### 1b. The 10 AQ-10 items

Each `A*_Score` should be 0 or 1 (yes/no on that question). Means by class show which items are more often 1 among YES rows — this is still the questionnaire, not a clinical biomarker.

In [ ]:
print(df[ITEM_COLS].describe().T[["min", "max", "mean"]])
print("\nitem value counts (should be 0/1):")
for c in ITEM_COLS:
    print(c, df[c].value_counts(dropna=False).to_dict())

item_means = df.groupby("class")[ITEM_COLS].mean().T
item_means.columns = [str(c) for c in item_means.columns]
print("\nmean(item = 1) by class")
print(item_means.round(3))

item_means.plot(kind="bar", figsize=(10, 4))
plt.ylabel("share answering 1")
plt.title("AQ-10 item endorsement by class")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[ITEM_COLS].corr(), cmap="vlag", center=0, annot=False)
plt.title("Correlation among the 10 items")
plt.tight_layout()
plt.show()

### 1c. Circularity check (label vs item total)

On this Combined 4–11 slice, YES is the AQ-10 total **≥ 7**. If that holds, any items-only model that looks “perfect” is copying the scoring rule. That is acceptable for a webpage that **explains AQ-10**, and it is **not** evidence of independent diagnosis.

In [ ]:
df["aq_sum"] = df[ITEM_COLS].sum(axis=1)
print(df.groupby("class")["aq_sum"].describe())
print("\ncrosstab item total vs class")
print(pd.crosstab(df["aq_sum"], df["class"], margins=True))

for cut in (6, 7):
    yes = df["class"] == "YES"
    agree = ((yes & (df["aq_sum"] >= cut)) | (~yes & (df["aq_sum"] < cut))).mean()
    print(f"agreement if YES meant total >= {cut}: {100 * agree:.1f}%")

plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="aq_sum", hue="class", multiple="dodge", discrete=True)
plt.axvline(6.5, color="black", linestyle="--", label="AQ-10 cutoff ~7")
plt.title("AQ item total by class (Thabtah Combined 4–11)")
plt.legend()
plt.tight_layout()
plt.show()

## 2. Clean-up + feature sets

- Drop `instrument` / `age_desc` (constant for this slice).
- Drop `aq_sum` so we do not feed the total that already defines the label.
- Normalise text fields.
- **Items** = A1–A10 → this is what the webpage sends.
- **Demographics** = age, gender, jaundice, family ASD → audit only.
- **Both** = check that background does not beat the questions.

In [ ]:
model_df = df.replace("?", np.nan).copy()
for col in ["gender", "jaundice", "autism"]:
    if col in model_df.columns:
        model_df[col] = model_df[col].astype("string").str.strip().str.lower().fillna("missing")

y = model_df["class"].map({"NO": 0, "YES": 1})
if y.isna().any():
    raise ValueError(model_df["class"].value_counts(dropna=False))

X_all = model_df.drop(columns=["class", "aq_sum", "instrument", "age_desc"], errors="ignore")
demo_cols = [c for c in X_all.columns if c not in ITEM_COLS]
print("Items (webpage inputs):", ITEM_COLS)
print("Background (audit only):", demo_cols)
print("X_all shape", X_all.shape, "YES rate", float(y.mean()))

## 3. Nested CV (four algorithms)

Outer 5-fold = the score we report. Inner 3-fold = pick `C` / depth / learning rate. Same four algorithms as the other questionnaire notebooks.

The **deployed `.pkl` is items-only logistic regression**, because the app only collects AQ-10 answers.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("train", X_train.shape, "test", X_test.shape, "test YES", int(y_test.sum()), "test NO", int((y_test == 0).sum()))


def make_preprocessor(feature_names):
    num = [c for c in feature_names if pd.api.types.is_numeric_dtype(X_all[c])]
    cat = [c for c in feature_names if c not in num]
    parts = []
    if num:
        parts.append(
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                num,
            )
        )
    if cat:
        parts.append(
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                cat,
            )
        )
    return ColumnTransformer(parts)


MODEL_GRIDS = {
    "log_reg": (
        LogisticRegression(max_iter=2000, class_weight="balanced"),
        {"model__C": [0.1, 1.0, 10.0]},
    ),
    "tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        {"model__max_depth": [2, 3, 4]},
    ),
    "rf": (
        RandomForestClassifier(
            random_state=RANDOM_STATE, class_weight="balanced", n_estimators=200
        ),
        {"model__max_depth": [3, 5]},
    ),
    "hgb": (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE, max_depth=3),
        {"model__learning_rate": [0.05, 0.1]},
    ),
}
FEATURE_SETS = {"items": ITEM_COLS, "demographics": demo_cols, "both": list(X_all.columns)}

outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
rows, fitted = [], {}
for set_name, cols in FEATURE_SETS.items():
    Xtr = X_train[cols]
    for model_name, (est, grid) in MODEL_GRIDS.items():
        gs = GridSearchCV(
            Pipeline([("preprocess", make_preprocessor(cols)), ("model", est)]),
            grid,
            cv=inner,
            scoring="roc_auc",
            n_jobs=1,
            refit=True,
        )
        aucs = []
        for tr_idx, te_idx in outer.split(Xtr, y_train):
            gs.fit(Xtr.iloc[tr_idx], y_train.iloc[tr_idx])
            aucs.append(roc_auc_score(y_train.iloc[te_idx], gs.predict_proba(Xtr.iloc[te_idx])[:, 1]))
        gs.fit(Xtr, y_train)
        fitted[(set_name, model_name)] = gs
        rows.append(
            {
                "feature_set": set_name,
                "model": model_name,
                "cv_mean": float(np.mean(aucs)),
                "cv_std": float(np.std(aucs)),
            }
        )
        print(f"{set_name:13} {model_name:8}  {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
cv_table = pd.DataFrame(rows).sort_values(["feature_set", "cv_mean"], ascending=[True, False])
cv_table

## 4. Holdout test (20%)

Sanity check on leftover children. Expect items / both ≈ perfect and demographics ≈ chance if the label is the AQ-10 cutoff.

In [ ]:
summary_rows = []
for set_name in FEATURE_SETS:
    best_name = cv_table[cv_table["feature_set"] == set_name].iloc[0]["model"]
    cols = FEATURE_SETS[set_name]
    gs = fitted[(set_name, best_name)]
    y_pred = gs.predict(X_test[cols])
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    summary_rows.append(
        {
            "feature_set": set_name,
            "model": best_name,
            "test_accuracy": (y_pred == y_test).mean(),
            "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        }
    )
    print("\n===", set_name, best_name, "===")
    print(classification_report(y_test, y_pred, target_names=["NO", "YES"]))
    print("confusion [[TN, FP], [FN, TP]]")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

screen_table = pd.DataFrame(summary_rows)
screen_table

In [ ]:
items_gs = fitted[("items", "log_reg")]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
cm = confusion_matrix(y_test, items_gs.predict(X_test[ITEM_COLS]), labels=[0, 1])
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["pred NO", "pred YES"],
    yticklabels=["true NO", "true YES"],
    ax=axes[0],
)
axes[0].set_title("Items log_reg — holdout confusion")
RocCurveDisplay.from_estimator(items_gs, X_test[ITEM_COLS], y_test, ax=axes[1])
axes[1].set_title("Items log_reg — holdout ROC")
plt.tight_layout()
plt.show()

## 5. SHAP — which questions push the flag

Fit items-only logistic regression (same inputs as the webpage). **Bar plot:** average |influence| of each AQ-10 item. **Beeswarm:** each holdout child; colour = item 0 or 1. **Waterfall:** one example YES and one example NO.

This belongs in the report / notebook. The first version of the webpage can skip SHAP and only call `.predict_proba`.

In [ ]:
try:
    import shap
except ImportError:
    import subprocess, sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])
    import shap

item_lr = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
item_lr.fit(X_train[ITEM_COLS], y_train)

explainer = shap.LinearExplainer(item_lr, X_train[ITEM_COLS])
shap_values = explainer(X_test[ITEM_COLS])

mean_abs = pd.Series(np.abs(shap_values.values).mean(axis=0), index=ITEM_COLS).sort_values(
    ascending=False
)
print("Average |SHAP| per AQ-10 item (holdout):")
print(mean_abs.round(4))

shap.plots.bar(shap_values, show=True)
shap.plots.beeswarm(shap_values, show=True)

In [ ]:
# One screen-positive and one screen-negative example on the holdout.
yes_pos = np.where(y_test.to_numpy() == 1)[0]
no_pos = np.where(y_test.to_numpy() == 0)[0]
print("example YES row (holdout iloc)", int(yes_pos[0]), "true label 1")
print(X_test[ITEM_COLS].iloc[yes_pos[0]].to_dict(), "sum", int(X_test[ITEM_COLS].iloc[yes_pos[0]].sum()))
shap.plots.waterfall(shap_values[int(yes_pos[0])], show=True)

print("example NO row (holdout iloc)", int(no_pos[0]), "true label 0")
print(X_test[ITEM_COLS].iloc[no_pos[0]].to_dict(), "sum", int(X_test[ITEM_COLS].iloc[no_pos[0]].sum()))
shap.plots.waterfall(shap_values[int(no_pos[0])], show=True)

## 6. Save `.pkl` for the webpage

Refit items-only logistic regression on **all 4–11 rows** after the holdout check. The pickle is a dict so the backend does not have to guess column names.

**Give your teammate:** `models/aq10_child_thabtah_4to11_logreg.pkl`

The form must send **exactly** these keys, each `0` or `1`:

`A1_Score` … `A10_Score`

In [ ]:
deploy_pipe = Pipeline(
    [
        (
            "preprocess",
            ColumnTransformer(
                [
                    (
                        "num",
                        Pipeline(
                            [
                                ("imputer", SimpleImputer(strategy="median")),
                                ("scaler", StandardScaler()),
                            ]
                        ),
                        ITEM_COLS,
                    )
                ]
            ),
        ),
        (
            "model",
            LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0),
        ),
    ]
)
deploy_pipe.fit(X_all[ITEM_COLS], y)

bundle = {
    "pipeline": deploy_pipe,
    "feature_names": ITEM_COLS,
    "target_names": ["NO", "YES"],
    "positive_class": "YES",
    "threshold": 0.5,
    "instrument": "AQ-10-Child",
    "age_band": "4-11",
    "n_rows_trained": int(len(X_all)),
    "sklearn_warning": "Load with the same sklearn version used to save this file.",
    "disclaimer": (
        "Screening aid only, not a diagnosis. Output is screen-positive / "
        "screen-negative from AQ-10-Child answers. Public labels follow the "
        "questionnaire cutoff."
    ),
}

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)
pkl_path = models_dir / "aq10_child_thabtah_4to11_logreg.pkl"
joblib.dump(bundle, pkl_path)

meta_path = models_dir / "aq10_child_thabtah_4to11_logreg_meta.json"
meta = {k: v for k, v in bundle.items() if k != "pipeline"}
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("wrote", pkl_path.resolve())
print("wrote", meta_path.resolve())
print("feature_names", ITEM_COLS)

## 7. How the webpage uses the `.pkl`

1. Put `aq10_child_thabtah_4to11_logreg.pkl` on the **backend** (Flask / FastAPI / Django). Do not load pickle in the browser.
2. The HTML form has 10 yes/no questions. Map Yes → `1`, No → `0`. Names must match `A1_Score` … `A10_Score`.
3. Backend builds a one-row pandas DataFrame with those columns, in that order, then:

```python
import joblib
import pandas as pd

bundle = joblib.load("models/aq10_child_thabtah_4to11_logreg.pkl")
pipe = bundle["pipeline"]
cols = bundle["feature_names"]

# example: body from the form / JSON API
answers = {"A1_Score": 1, "A2_Score": 1, "A3_Score": 0, "A4_Score": 1,
           "A5_Score": 1, "A6_Score": 1, "A7_Score": 1, "A8_Score": 0,
           "A9_Score": 1, "A10_Score": 1}
row = pd.DataFrame([answers], columns=cols)
proba = float(pipe.predict_proba(row)[0, 1])       # P(screen-positive)
label = "YES" if proba >= bundle["threshold"] else "NO"
```

4. Show the user: **screen-positive or screen-negative**, the probability, and `bundle["disclaimer"]`.
5. Same sklearn version as this notebook (see `pip show scikit-learn`).

The next cell is a local smoke test of that contract.

In [ ]:
loaded = joblib.load(pkl_path)
pipe = loaded["pipeline"]
cols = loaded["feature_names"]

demo_yes = {c: 1 for c in cols}          # all yes → should be screen-positive
demo_no = {c: 0 for c in cols}           # all no  → should be screen-negative
for name, answers in [("all_1", demo_yes), ("all_0", demo_no)]:
    row = pd.DataFrame([answers], columns=cols)
    proba = float(pipe.predict_proba(row)[0, 1])
    pred = pipe.predict(row)[0]
    print(name, "P(YES)=", round(proba, 3), "label=", loaded["target_names"][int(pred)])

print("\n--- copy this JSON shape for the web API ---")
print(json.dumps({"answers": {c: 0 for c in cols}}, indent=2))

## How to read the modelling

| Result | Meaning for the app |
|---|---|
| Items log_reg nested-CV AUC ≈ 1.0 | The `.pkl` implements AQ-10 scoring, which is what the form asks |
| Demographics AUC ≈ 0.5 | Do **not** send age/gender into this pickle |
| SHAP bars | Which of the 10 questions moved the score (report / later UI) |
| 100% holdout accuracy | Expected on this labelled public file — **do not** print it on the website as “autism detection accuracy” |